# Sentinel — 02: DuckDB Yükleme ve İlk Sorgular
**Amaç:** CheckUp Ana + Lab + Rec verilerini DuckDB'ye yükleyip Sentinel için hazır hale getirmek.

**Sıra:**
1. DuckDB bağlantısı kur
2. CheckUp Ana (6 dosya) → `ana` tablosu
3. CheckUp Lab (72 dosya) → `lab` tablosu
4. CheckUp Rec (4 dosya) → `rec` tablosu
5. Doğrulama sorguları
6. İlk Sentinel hasta profili

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'duckdb'], check=True)


CompletedProcess(args=['pip', 'install', 'duckdb'], returncode=0)

In [2]:
import duckdb
import pandas as pd
import numpy as np
import os
import glob
import time
import warnings
warnings.filterwarnings('ignore')

print('✅ Kütüphaneler yüklendi')
print(f'DuckDB version: {duckdb.__version__}')

✅ Kütüphaneler yüklendi
DuckDB version: 1.4.4


---
## ⚙️ BÖLÜM 0: Yol Ayarları

In [3]:
# ── BURASI SİZE ÖZEL — DEĞİŞTİRMEYİN SADECE BASE_PATH'İ ──
BASE_PATH = r"C:\Users\Mutlu\Desktop\ACUHIT\data"
DB_PATH   = r"C:\Users\Mutlu\Desktop\ACUHIT\sentinel.db"

ANA_PATH = os.path.join(BASE_PATH, "Check-Up - Data", "Check_Up_Anadata")
LAB_PATH = os.path.join(BASE_PATH, "Check-Up - Data", "Check_Up_Lab")
REC_PATH = os.path.join(BASE_PATH, "Check-Up - Data", "Check_Up_Recete")

# Dosyaları listele
ana_files = sorted(glob.glob(os.path.join(ANA_PATH, "*.csv")))
lab_files = sorted(glob.glob(os.path.join(LAB_PATH, "*.csv")))
rec_files = sorted(glob.glob(os.path.join(REC_PATH, "*.csv")))

print(f'Ana  dosyaları: {len(ana_files)} adet')
print(f'Lab  dosyaları: {len(lab_files)} adet')
print(f'Rec  dosyaları: {len(rec_files)} adet')
print(f'DB   konumu   : {DB_PATH}')

Ana  dosyaları: 6 adet
Lab  dosyaları: 72 adet
Rec  dosyaları: 4 adet
DB   konumu   : C:\Users\Mutlu\Desktop\ACUHIT\sentinel.db


---
## 🗄️ BÖLÜM 1: DuckDB Bağlantısı

In [4]:
con = duckdb.connect(DB_PATH)

# Performans ayarları
con.execute("SET threads=4")
con.execute("SET memory_limit='4GB'")

print(f'✅ DuckDB bağlantısı kuruldu: {DB_PATH}')
print('Mevcut tablolar:')
mevcut = con.execute("SHOW TABLES").fetchdf()
if len(mevcut) == 0:
    print('  (henüz tablo yok)')
else:
    print(mevcut.to_string())

✅ DuckDB bağlantısı kuruldu: C:\Users\Mutlu\Desktop\ACUHIT\sentinel.db
Mevcut tablolar:
  name
0  ana
1  lab
2  rec


---
## 📋 BÖLÜM 2: CheckUp Ana Yükleme (6 dosya)

In [5]:
print('=== ANA VERİ YÜKLENİYOR ===')
t0 = time.time()

con.execute("DROP TABLE IF EXISTS ana")

# DuckDB'nin read_csv_auto ile direkt yükle — pandas'a gerek yok
ana_glob = os.path.join(ANA_PATH, "*.csv").replace("\\", "/")

con.execute(f"""
    CREATE TABLE ana AS
    SELECT * FROM read_csv_auto(
        '{ana_glob}',
        header=true,
        ignore_errors=true,
        sample_size=10000
    )
""")

# İndeks ekle
con.execute("CREATE INDEX IF NOT EXISTS idx_ana_hasta ON ana(HASTA_ID)")
con.execute("CREATE INDEX IF NOT EXISTS idx_ana_episode ON ana(SQ_EPISODE)")

# Kontrol
n_rows = con.execute("SELECT COUNT(*) FROM ana").fetchone()[0]
n_hasta = con.execute("SELECT COUNT(DISTINCT HASTA_ID) FROM ana").fetchone()[0]
t1 = time.time()

print(f'✅ Ana tablosu hazır!')
print(f'   Toplam satır : {n_rows:,}')
print(f'   Benzersiz hasta: {n_hasta:,}')
print(f'   Süre: {t1-t0:.1f} saniye')

=== ANA VERİ YÜKLENİYOR ===
✅ Ana tablosu hazır!
   Toplam satır : 5,887,648
   Benzersiz hasta: 210,304
   Süre: 60.4 saniye


In [8]:
# Ana tablo yapısını kontrol et
print('Ana tablo sütunları:')
ana_cols = con.execute("DESCRIBE ana").fetchdf()
print(ana_cols[['column_name','column_type']].to_string())

Ana tablo sütunları:
                    column_name column_type
0                      HASTA_ID     VARCHAR
1                   RF_EPISODE2      BIGINT
2                    SQ_EPISODE      BIGINT
3                 EPISODE_TARIH   TIMESTAMP
4           TOPLAM_GELIS_SAYISI      BIGINT
5                  GELIS_SAYISI      BIGINT
6   ILK_TANI_SON_TANI_GUN_FARKI      DOUBLE
7                     TANI_YASI      BIGINT
8                      CINSIYET     VARCHAR
9                     TANITARIH   TIMESTAMP
10               MIN_TANI_TARIH   TIMESTAMP
11               MAX_TANI_TARIH   TIMESTAMP
12                     TANIKODU     VARCHAR
13                    TANI_TIPI     VARCHAR
14                   GELIS_TIPI     VARCHAR
15                    SERVISADI     VARCHAR
16                      YAKINMA     VARCHAR
17                         ÖYKÜ     VARCHAR
18             BASLANGIC_ZAMANI   TIMESTAMP
19     YAKINMA_BASLANGIC_ZAMANI   TIMESTAMP
20                  Düşme Riski     VARCHAR
21         

---
## 🧪 BÖLÜM 3: CheckUp Lab Yükleme (72 dosya — 72M satır)


In [6]:
print('=== LAB VERİSİ YÜKLENİYOR (72 dosya × ~1M satır) ===')
print('Bu işlem 3-10 dakika sürebilir...')
t0 = time.time()

con.execute("DROP TABLE IF EXISTS lab")

lab_glob = os.path.join(LAB_PATH, "*.csv").replace("\\", "/")

con.execute(f"""
    CREATE TABLE lab AS
    SELECT
        HASTA_ID,
        SUB_CODE,
        RESULT,
        TRY_CAST(RESULT AS DOUBLE) AS RESULT_NUM,
        UNIT,
        TRY_CAST(
            REGEXP_REPLACE(CAST(REFMIN AS VARCHAR), '[^0-9.]', '', 'g')
        AS DOUBLE) AS REFMIN_NUM,
        TRY_CAST(
            REGEXP_REPLACE(CAST(REFMAX AS VARCHAR), '[^0-9.]', '', 'g')
        AS DOUBLE) AS REFMAX_NUM,
        CAST(REP_DATE AS TIMESTAMP) AS REP_DATE
    FROM read_csv_auto(
        '{lab_glob}',
        header=true,
        ignore_errors=true,
        sample_size=5000
    )
""")

# Anormallik kolonu ekle
con.execute("""
    ALTER TABLE lab ADD COLUMN IF NOT EXISTS is_abnormal BOOLEAN;
    UPDATE lab SET is_abnormal = (
        RESULT_NUM IS NOT NULL AND
        REFMIN_NUM IS NOT NULL AND
        REFMAX_NUM IS NOT NULL AND
        (RESULT_NUM < REFMIN_NUM OR RESULT_NUM > REFMAX_NUM)
    )
""")

# İndeks
con.execute("CREATE INDEX IF NOT EXISTS idx_lab_hasta   ON lab(HASTA_ID)")
con.execute("CREATE INDEX IF NOT EXISTS idx_lab_subcode ON lab(SUB_CODE)")
con.execute("CREATE INDEX IF NOT EXISTS idx_lab_date    ON lab(REP_DATE)")

n_rows  = con.execute("SELECT COUNT(*) FROM lab").fetchone()[0]
n_hasta = con.execute("SELECT COUNT(DISTINCT HASTA_ID) FROM lab").fetchone()[0]
n_tests = con.execute("SELECT COUNT(DISTINCT SUB_CODE) FROM lab").fetchone()[0]
t1 = time.time()

print(f'✅ Lab tablosu hazır!')
print(f'   Toplam satır    : {n_rows:,}')
print(f'   Benzersiz hasta : {n_hasta:,}')
print(f'   Benzersiz test  : {n_tests:,}')
print(f'   Süre: {t1-t0:.1f} saniye')

=== LAB VERİSİ YÜKLENİYOR (72 dosya × ~1M satır) ===
Bu işlem 3-10 dakika sürebilir...
✅ Lab tablosu hazır!
   Toplam satır    : 71,101,425
   Benzersiz hasta : 239,263
   Benzersiz test  : 6,193
   Süre: 370.2 saniye


---
## 💊 BÖLÜM 4: CheckUp Rec Yükleme (4 dosya)

In [7]:
print('=== REÇETE VERİSİ YÜKLENİYOR ===')
t0 = time.time()

con.execute("DROP TABLE IF EXISTS rec")

rec_glob = os.path.join(REC_PATH, "*.csv").replace("\\", "/")

con.execute(f"""
    CREATE TABLE rec AS
    SELECT * FROM read_csv_auto(
        '{rec_glob}',
        header=true,
        ignore_errors=true,
        sample_size=5000
    )
""")

con.execute("CREATE INDEX IF NOT EXISTS idx_rec_hasta   ON rec(HASTA_ID)")
con.execute("CREATE INDEX IF NOT EXISTS idx_rec_episode ON rec(RF_EPISODE)")

n_rows  = con.execute("SELECT COUNT(*) FROM rec").fetchone()[0]
n_hasta = con.execute("SELECT COUNT(DISTINCT HASTA_ID) FROM rec").fetchone()[0]
t1 = time.time()

print(f'✅ Rec tablosu hazır!')
print(f'   Toplam satır    : {n_rows:,}')
print(f'   Benzersiz hasta : {n_hasta:,}')
print(f'   Süre: {t1-t0:.1f} saniye')

=== REÇETE VERİSİ YÜKLENİYOR ===
✅ Rec tablosu hazır!
   Toplam satır    : 3,551,983
   Benzersiz hasta : 207,434
   Süre: 18.3 saniye


---
## ✅ BÖLÜM 5: Doğrulama Sorguları

In [8]:
print('=== VERİTABANI ÖZET ===')
tablolar = ['ana', 'lab', 'rec']
for tablo in tablolar:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {tablo}").fetchone()[0]
        nh = con.execute(f"SELECT COUNT(DISTINCT HASTA_ID) FROM {tablo}").fetchone()[0]
        ncols = len(con.execute(f"DESCRIBE {tablo}").fetchdf())
        print(f'  {tablo:<6}: {n:>12,} satır | {nh:>8,} hasta | {ncols} sütun')
    except Exception as e:
        print(f'  {tablo}: HATA — {e}')

=== VERİTABANI ÖZET ===
  ana   :    5,887,648 satır |  210,304 hasta | 61 sütun
  lab   :   71,101,425 satır |  239,263 hasta | 9 sütun
  rec   :    3,551,983 satır |  207,434 hasta | 9 sütun


In [9]:
# En çok ziyareti olan 10 hasta
print('=== EN FAZLA ZİYARETİ OLAN 10 HASTA ===')
df = con.execute("""
    SELECT
        HASTA_ID,
        COUNT(*) AS ziyaret_sayisi,
        MAX(TOPLAM_GELIS_SAYISI) AS toplam_gelis,
        MAX(TANI_YASI) AS yas,
        MAX(CINSIYET) AS cinsiyet,
        COUNT(DISTINCT SERVISADI) AS n_bolum
    FROM ana
    GROUP BY HASTA_ID
    ORDER BY ziyaret_sayisi DESC
    LIMIT 10
""").fetchdf()
print(df.to_string(index=False))

=== EN FAZLA ZİYARETİ OLAN 10 HASTA ===
   HASTA_ID  ziyaret_sayisi  toplam_gelis  yas cinsiyet  n_bolum
       None           65733           118   82        K       83
ANON_234159            5119            27   72        E       13
ANON_200081            3890            76   51        K       17
ANON_231822            2608           110   40        K       16
ANON_200752            2464            33   59        K        9
ANON_189904            2305            68   42        K       14
ANON_190415            1417            63   49        K        8
ANON_136224            1392           832   42        K       28
ANON_232227            1313            44   46        K       19
ANON_082030            1230            10   40        K        5


In [10]:
# Lab'da en çok ölçülen 20 test
print('=== EN SIK 20 LAB TESTİ ===')
df = con.execute("""
    SELECT
        SUB_CODE,
        COUNT(*) AS olcum_sayisi,
        ROUND(AVG(CASE WHEN is_abnormal THEN 1.0 ELSE 0.0 END) * 100, 1) AS anormal_pct
    FROM lab
    GROUP BY SUB_CODE
    ORDER BY olcum_sayisi DESC
    LIMIT 20
""").fetchdf()
print(df.to_string(index=False))

=== EN SIK 20 LAB TESTİ ===
     SUB_CODE  olcum_sayisi  anormal_pct
      Lökosit       1075118         17.5
    Eritrosit       1058146         14.8
       RDW-CV       1010521         10.3
   Hematokrit       1010506         27.0
 Nötrofil (%)       1010502         23.9
   Hemoglobin       1010500         25.7
 Nötrofil (#)       1010499         18.4
 Lenfosit (#)       1010496         21.1
    Trombosit       1010494          7.0
         MCHC       1010491          5.5
 Lenfosit (%)       1010483         23.4
          MCH       1010483         12.2
          MCV       1010472         17.4
          MPV       1010416          3.6
          PDW       1010396          6.9
  Monosit (#)       1008891         16.2
  Monosit (%)       1008886         10.5
Eozinofil (#)       1007902          2.3
Eozinofil (%)       1007897          8.7
  Bazofil (%)       1007888          2.1


In [11]:
# Ana ↔ Lab join testi
print('=== JOIN DOĞRULAMA ===')
df = con.execute("""
    SELECT
        COUNT(DISTINCT a.HASTA_ID) AS ana_hasta,
        COUNT(DISTINCT l.HASTA_ID) AS lab_hasta,
        COUNT(DISTINCT CASE WHEN l.HASTA_ID IS NOT NULL THEN a.HASTA_ID END) AS kesisim
    FROM ana a
    LEFT JOIN lab l ON a.HASTA_ID = l.HASTA_ID
""").fetchdf()
print(df.to_string(index=False))

=== JOIN DOĞRULAMA ===
 ana_hasta  lab_hasta  kesisim
    210304     184975   184975


---
## 🎯 BÖLÜM 6: İlk Sentinel Hasta Profili
Demo için en uygun hastayı seçiyoruz: çok ziyaret, çok lab, risk var.

In [12]:


import pandas as pd
import os

LAB_PATH = r"C:\Users\Mutlu\Desktop\ACUHIT\data\Check-Up - Data\Check_Up_Lab"

# Sadece ilk dosyadan 50K satır al
df_sample = pd.read_csv(
    os.path.join(LAB_PATH, "lab_0001.csv"),
    nrows=50000
)

# Anormal olanları bul
df_sample['RESULT_NUM'] = pd.to_numeric(df_sample['RESULT'], errors='coerce')
df_sample['REFMIN_NUM'] = pd.to_numeric(df_sample['REFMIN'], errors='coerce')
df_sample['REFMAX_NUM'] = pd.to_numeric(df_sample['REFMAX'], errors='coerce')
df_sample['is_abnormal'] = (
    (df_sample['RESULT_NUM'] < df_sample['REFMIN_NUM']) |
    (df_sample['RESULT_NUM'] > df_sample['REFMAX_NUM'])
)

# En fazla anormal labı olan hasta
demo = (df_sample.groupby('HASTA_ID')
        .agg(n_lab=('RESULT','count'),
             n_anormal=('is_abnormal','sum'))
        .query('n_lab > 10 and n_anormal > 3')
        .sort_values('n_anormal', ascending=False)
        .head(5))

print(demo)
DEMO_HASTA = demo.index[0]
print(f'\nDemo hasta: {DEMO_HASTA}')

             n_lab  n_anormal
HASTA_ID                     
ANON_018774   3973        502
ANON_050838   1741        208
ANON_048589    445        119
ANON_021367    719        116
ANON_073301    879        112

Demo hasta: ANON_018774


In [13]:
import duckdb
con = duckdb.connect(r"C:\Users\Mutlu\Desktop\ACUHIT\sentinel.db")

DEMO_HASTA = 'ANON_018774'

# 1. Temel bilgiler
print('=== TEMEL BİLGİLER ===')
temel = con.execute(f"""
    SELECT
        HASTA_ID, MAX(TANI_YASI) AS yas, MAX(CINSIYET) AS cinsiyet,
        COUNT(DISTINCT SQ_EPISODE) AS n_ziyaret,
        COUNT(DISTINCT SERVISADI) AS n_bolum,
        MIN(EPISODE_TARIH) AS ilk, MAX(EPISODE_TARIH) AS son
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY HASTA_ID
""").fetchdf()
print(temel.to_string(index=False))

# 2. Kritik lab testleri
print('\n=== KRİTİK LAB TESTLERİ ===')
lab = con.execute(f"""
    SELECT SUB_CODE,
        COUNT(*) AS n,
        ROUND(MIN(RESULT_NUM),2) AS min_val,
        ROUND(MAX(RESULT_NUM),2) AS max_val,
        ROUND(AVG(RESULT_NUM),2) AS ort,
        SUM(CASE WHEN is_abnormal THEN 1 ELSE 0 END) AS n_anormal,
        MAX(REP_DATE) AS son_tarih
    FROM lab
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND RESULT_NUM IS NOT NULL
    GROUP BY SUB_CODE
    ORDER BY n_anormal DESC
    LIMIT 15
""").fetchdf()
print(lab.to_string(index=False))

# 3. Reçete geçmişi
print('\n=== REÇETE GEÇMİŞİ ===')
rec = con.execute(f"""
    SELECT "İlaç Adı", COUNT(*) AS n,
        MIN(RECETE_TARIH) AS ilk,
        MAX(RECETE_TARIH) AS son
    FROM rec
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY "İlaç Adı"
    ORDER BY n DESC
    LIMIT 15
""").fetchdf()
print(rec.to_string(index=False))

# 4. Metin notları
print('\n=== SON 3 ZİYARET NOTLARI ===')
notlar = con.execute(f"""
    SELECT EPISODE_TARIH, SERVISADI, YAKINMA,
           "Muayene Notu", "Tedavi Notu", "Özgeçmiş Notu",
           "Sürekli Kullandığı İlaçlar"
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND ("Muayene Notu" IS NOT NULL OR YAKINMA IS NOT NULL)
    ORDER BY EPISODE_TARIH DESC
    LIMIT 3
""").fetchdf()
for _, row in notlar.iterrows():
    print(f"\n[{row['EPISODE_TARIH']} — {row['SERVISADI']}]")
    for col in ['YAKINMA','Muayene Notu','Tedavi Notu','Özgeçmiş Notu','Sürekli Kullandığı İlaçlar']:
        val = row.get(col)
        if str(val) not in ['nan','None',''] and val == val:
            print(f"  {col}: {str(val)[:120]}")

=== TEMEL BİLGİLER ===
   HASTA_ID  yas cinsiyet  n_ziyaret  n_bolum                 ilk                 son
ANON_018774   81        E        165       15 2009-11-03 03:38:23 2025-05-10 11:55:55

=== KRİTİK LAB TESTLERİ ===
                            SUB_CODE  n  min_val  max_val    ort  n_anormal           son_tarih
                              RDW-SD 59    41.80    52.90  47.53       52.0 2025-05-05 20:57:38
                       Glukoz, Açlık 33   104.67   217.59 153.67       33.0 2025-05-05 22:05:10
                     Üre Azotu (BUN) 74    11.00    40.00  22.61       30.0 2025-05-05 21:37:01
                        Lenfosit (%) 59     3.30    37.40  19.89       30.0 2025-05-05 20:57:36
                             Albumin 34     2.40     4.17   2.93       27.0 2023-12-23 04:16:21
                        Lenfosit (#) 59     0.27     3.14   1.40       25.0 2025-05-05 20:57:35
                        Nötrofil (%) 59    51.20    92.90  70.00       25.0 2025-05-05 20:57:22
Eritrosi

In [14]:
# Reçete tam liste + NSAİİ/risk ilaç kontrolü
print('=== REÇETE TAM LİSTE ===')
rec_full = con.execute(f"""
    SELECT "İlaç Adı", COUNT(*) AS n,
        MIN(RECETE_TARIH) AS ilk,
        MAX(RECETE_TARIH) AS son
    FROM rec
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY "İlaç Adı"
    ORDER BY n DESC
""").fetchdf()
print(rec_full.to_string(index=False))

# Hemoglobin trend — kritik!
print('\n=== HEMOGLOBİN TRENDİ ===')
hb = con.execute(f"""
    SELECT CAST(REP_DATE AS DATE) AS tarih,
           RESULT_NUM AS hb,
           REFMIN_NUM AS ref_min,
           REFMAX_NUM AS ref_max,
           is_abnormal
    FROM lab
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND SUB_CODE = 'Hemoglobin'
      AND RESULT_NUM IS NOT NULL
    ORDER BY REP_DATE
""").fetchdf()
print(hb.to_string(index=False))

# Glukoz trend
print('\n=== GLUKOZ TRENDİ ===')
glu = con.execute(f"""
    SELECT CAST(REP_DATE AS DATE) AS tarih,
           RESULT_NUM AS glukoz,
           REFMAX_NUM AS ref_max,
           is_abnormal
    FROM lab
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND SUB_CODE LIKE '%Glukoz%'
      AND RESULT_NUM IS NOT NULL
    ORDER BY REP_DATE
""").fetchdf()
print(glu.to_string(index=False))

# Tüm metin notları
print('\n=== TÜM BÖLÜMLER ===')
bolum = con.execute(f"""
    SELECT DISTINCT SERVISADI,
        COUNT(*) AS n_ziyaret
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY SERVISADI
    ORDER BY n_ziyaret DESC
""").fetchdf()
print(bolum.to_string(index=False))

=== REÇETE TAM LİSTE ===
                               İlaç Adı  n                 ilk                 son
JANUMET 50/1000 MG 56 FILM KAPLI TABLET  1 2022-10-14 17:18:11 2022-10-14 17:18:11
              CEFAKS 500 MG.10 FILM TB.  1 2025-05-10 11:54:28 2025-05-10 11:54:28
                   BACTRIM FORTE 20 TB.  1 2025-07-29 07:56:10 2025-07-29 07:56:10
    ACE PLUS SELENYUM 30 YUMUSAK KAPSUL  1 2022-10-14 17:18:11 2022-10-14 17:18:11
      JARDIANCE 10 MG FILM KAPLI TABLET  1 2022-10-14 17:18:11 2022-10-14 17:18:11
                   BUTEFIN %1 KREM 30 G  1 2022-04-05 12:26:13 2022-04-05 12:26:13
                 EXELDERM %1 30 GR KREM  1 2022-04-05 12:26:13 2022-04-05 12:26:13
                     DODEX 1 CC.5 AMPUL  1 2022-10-14 17:18:11 2022-10-14 17:18:11
               MAGNEZINC 40 FILM TABLET  1 2022-10-14 17:18:11 2022-10-14 17:18:11
    CLEXANE 4000 ANTI-XA 10 KUL.HAZ.ENJ  1 2025-05-10 11:54:28 2025-05-10 11:54:28
          CASODEX 150 MG.28 FILM TABLET  1 2025-05-22 16:20:10

In [15]:
print('=== GLUKOZ TRENDİ TAM ===')
glu = con.execute(f"""
    SELECT CAST(REP_DATE AS DATE) AS tarih,
           RESULT_NUM AS glukoz,
           REFMAX_NUM AS ref_max,
           is_abnormal
    FROM lab
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND SUB_CODE LIKE '%Glukoz%'
      AND RESULT_NUM IS NOT NULL
    ORDER BY REP_DATE
""").fetchdf()
print(glu.to_string(index=False))

print('\n=== TÜM BÖLÜMLER ===')
bolum = con.execute(f"""
    SELECT SERVISADI, COUNT(*) AS n_ziyaret
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY SERVISADI
    ORDER BY n_ziyaret DESC
""").fetchdf()
print(bolum.to_string(index=False))

print('\n=== SON 5 ZİYARET NOTU ===')
notlar = con.execute(f"""
    SELECT EPISODE_TARIH, SERVISADI,
           YAKINMA, "Muayene Notu", "Tedavi Notu",
           "Özgeçmiş Notu", "Sürekli Kullandığı İlaçlar"
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
    ORDER BY EPISODE_TARIH DESC
    LIMIT 5
""").fetchdf()
for _, row in notlar.iterrows():
    print(f"\n[{row['EPISODE_TARIH']} — {row['SERVISADI']}]")
    for col in ['YAKINMA','Muayene Notu','Tedavi Notu',
                'Özgeçmiş Notu','Sürekli Kullandığı İlaçlar']:
        val = row.get(col)
        if str(val) not in ['nan','None',''] and val == val:
            print(f"  {col}: {str(val)[:150]}")

=== GLUKOZ TRENDİ TAM ===
     tarih   glukoz  ref_max  is_abnormal
2006-09-15 158.0000    115.0         True
2006-11-23 162.9556    115.0         True
2007-02-22 161.0000    115.0         True
2007-09-04 217.5891    100.0         True
2007-09-21 136.9648    100.0         True
2008-02-08 177.0000    100.0         True
2008-05-09 152.6097    100.0         True
2008-06-25 194.1089    100.0         True
2009-02-03 130.3491    100.0         True
2009-07-11 136.2878    100.0         True
2009-08-05 166.9969    100.0         True
2010-01-06 121.6672    100.0         True
2010-03-20 164.9810    100.0         True
2010-09-15 167.6312    100.0         True
2011-01-11 113.1829    100.0         True
2012-01-23 212.6706    100.0         True
2012-03-30 151.9040      NaN        False
2012-05-11 104.6700    100.0         True
2012-07-14 172.0900    100.0         True
2012-12-13 157.8400    100.0         True
2012-12-13 174.8640      NaN        False
2013-01-24 155.6350      NaN        False
2013-03-

In [16]:
# Hastanın lab özeti
print(f'\n-- Lab Özeti: {DEMO_HASTA} --')
lab_ozet = con.execute(f"""
    SELECT
        SUB_CODE,
        COUNT(*) AS n_olcum,
        ROUND(MIN(RESULT_NUM), 2) AS min_val,
        ROUND(MAX(RESULT_NUM), 2) AS max_val,
        ROUND(AVG(RESULT_NUM), 2) AS ort_val,
        SUM(CASE WHEN is_abnormal THEN 1 ELSE 0 END) AS n_anormal,
        MAX(REP_DATE) AS son_tarih
    FROM lab
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY SUB_CODE
    ORDER BY n_anormal DESC, n_olcum DESC
    LIMIT 15
""").fetchdf()
print(lab_ozet.to_string(index=False))


-- Lab Özeti: ANON_018774 --
                            SUB_CODE  n_olcum  min_val  max_val  ort_val  n_anormal           son_tarih
                              RDW-SD       60    41.80    52.90    47.53       52.0 2025-05-05 20:57:38
                       Glukoz, Açlık       33   104.67   217.59   153.67       33.0 2025-05-05 22:05:10
                     Üre Azotu (BUN)       74    11.00    40.00    22.61       30.0 2025-05-05 21:37:01
                        Lenfosit (%)       60     3.30    37.40    19.89       30.0 2025-05-05 20:57:36
                             Albumin       34     2.40     4.17     2.93       27.0 2023-12-23 04:16:21
                        Lenfosit (#)       60     0.27     3.14     1.40       25.0 2025-05-05 20:57:35
                        Nötrofil (%)       60    51.20    92.90    70.00       25.0 2025-05-05 20:57:22
                          Hematokrit       60    34.40    51.30    44.45       20.0 2025-05-05 20:57:25
Eritrosit dağılım genişliği (RDW-S

In [17]:
# Hastanın reçete geçmişi
print(f'\n-- Reçete Geçmişi: {DEMO_HASTA} --')
rec_ozet = con.execute(f"""
    SELECT
        "İlaç Adı",
        COUNT(*) AS n_recete,
        MIN(RECETE_TARIH) AS ilk,
        MAX(RECETE_TARIH) AS son
    FROM rec
    WHERE HASTA_ID = '{DEMO_HASTA}'
    GROUP BY "İlaç Adı"
    ORDER BY n_recete DESC
    LIMIT 15
""").fetchdf()
print(rec_ozet.to_string(index=False))


-- Reçete Geçmişi: ANON_018774 --
                               İlaç Adı  n_recete                 ilk                 son
JANUMET 50/1000 MG 56 FILM KAPLI TABLET         1 2022-10-14 17:18:11 2022-10-14 17:18:11
              CEFAKS 500 MG.10 FILM TB.         1 2025-05-10 11:54:28 2025-05-10 11:54:28
                   BACTRIM FORTE 20 TB.         1 2025-07-29 07:56:10 2025-07-29 07:56:10
                   BUTEFIN %1 KREM 30 G         1 2022-04-05 12:26:13 2022-04-05 12:26:13
                 EXELDERM %1 30 GR KREM         1 2022-04-05 12:26:13 2022-04-05 12:26:13
                     DODEX 1 CC.5 AMPUL         1 2022-10-14 17:18:11 2022-10-14 17:18:11
               MAGNEZINC 40 FILM TABLET         1 2022-10-14 17:18:11 2022-10-14 17:18:11
    CLEXANE 4000 ANTI-XA 10 KUL.HAZ.ENJ         1 2025-05-10 11:54:28 2025-05-10 11:54:28
          CASODEX 150 MG.28 FILM TABLET         1 2025-05-22 16:20:10 2025-05-22 16:20:10
    ACE PLUS SELENYUM 30 YUMUSAK KAPSUL         1 2022-10-14 17:1

In [18]:
# Hastanın metin notları — NLP için
print(f'\n-- Metin Notları: {DEMO_HASTA} --')
notlar = con.execute(f"""
    SELECT
        EPISODE_TARIH,
        SERVISADI,
        YAKINMA,
        ÖYKÜ,
        "Muayene Notu",
        "Tedavi Notu",
        "Özgeçmiş Notu",
        "Sürekli Kullandığı İlaçlar"
    FROM ana
    WHERE HASTA_ID = '{DEMO_HASTA}'
      AND (YAKINMA IS NOT NULL OR "Muayene Notu" IS NOT NULL)
    ORDER BY EPISODE_TARIH DESC
    LIMIT 5
""").fetchdf()

for _, row in notlar.iterrows():
    print(f"\n[{row['EPISODE_TARIH']} — {row['SERVISADI']}]")
    for col in ['YAKINMA','ÖYKÜ','Muayene Notu','Tedavi Notu','Özgeçmiş Notu','Sürekli Kullandığı İlaçlar']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip():
            print(f"  {col}: {str(val)[:100]}")


-- Metin Notları: ANON_018774 --

[2025-05-10 11:55:55 — Üroloji]
  YAKINMA: ALT ÜRİNER SİSTEM YAKINMALARI VAR. SON 1 AYDA ORTA
  ÖYKÜ: ALT ÜRİNER SİSTEM YAKINMALARI VAR. SON 1 AYDA ORTALAMA NOKTÜRİ 2 KEZ OLMUŞ. YİNE 1 AYDIR DM REGÜLASY
  Muayene Notu: UF: 12/8/232, PMR 118 ML, KESİKLİ İDRAR AKIMI
PSA: 2.7
USG: PROSTAT 54 GR.
  Tedavi Notu: XATRAL XL TB - 1 AY 
UF TEKRARINA GÖRE MEDİKAL /CERRAHİ TEDAVİ KARARI VERİLECEK

[2025-05-10 11:55:55 — Üroloji]
  Muayene Notu: N
  Tedavi Notu: kinzy pyeloseptyl

[2025-05-10 11:55:55 — Üroloji]
  Muayene Notu: N
  Tedavi Notu: ciproxin

[2025-05-10 11:55:55 — Üroloji]
  Muayene Notu: Prostat grade1 benign
  Tedavi Notu: BPH nedeniyle Lazer prostatektomi (PVP) yapılması planlanmaktadır.

[2025-05-10 11:55:55 — Üroloji]
  Muayene Notu: Her iki lomber bölge simetrik.Böbrekler nonpalpabl.Bilateral kostovertebral açı hassasiyeti alınmadı


In [19]:
# Kritik lab trendleri — Sentinel motoru için
print(f'\n-- Kritik Lab Trendleri: {DEMO_HASTA} --')

KRITIK_TESTLER = [
    'Kreatinin', 'Hemoglobin', 'Glukoz', 'HbA1c',
    'Potasyum', 'Kolesterol', 'CRP', 'TSH'
]

for test in KRITIK_TESTLER:
    trend = con.execute(f"""
        SELECT
            CAST(REP_DATE AS DATE) AS tarih,
            RESULT_NUM AS deger,
            REFMIN_NUM AS ref_min,
            REFMAX_NUM AS ref_max,
            is_abnormal
        FROM lab
        WHERE HASTA_ID = '{DEMO_HASTA}'
          AND UPPER(SUB_CODE) LIKE UPPER('%{test}%')
          AND RESULT_NUM IS NOT NULL
        ORDER BY REP_DATE
    """).fetchdf()

    if len(trend) >= 2:
        vals   = trend['deger'].tolist()
        son    = vals[-1]
        ilk    = vals[0]
        degisim = son - ilk
        yon    = '↑ ARTIŞ' if degisim > 0 else '↓ DÜŞÜŞ'
        anorm  = '⚠️ ANORMAL' if trend['is_abnormal'].any() else '✅'
        print(f"  {test:<15}: {[round(v,1) for v in vals]}  {yon}  {anorm}")

print('\n✅ Bölüm 6 tamamlandı — demo hasta profili hazır!')


-- Kritik Lab Trendleri: ANON_018774 --
  Kreatinin      : [0.9, 0.9, 0.8, 0.8, 0.8, 0.9, 0.8, 0.9, 0.8, 0.8, 0.9, 0.9, 0.8, 0.9, 0.9, 0.9, 0.9, 0.9, 0.8, 1.0, 1.0, 0.8, 0.9, 1.1, 1.1, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.1, 1.1, 1.1, 1.1, 1.0, 1.0, 1.0, 0.9, 1.1, 1.1, 0.8, 0.9, 0.9, 0.8, 0.8, 0.8, 0.8, 1.6, 1.6, 0.9, 1.2, 1.1, 1.3, 1.3, 1.0, 1.0, 1.1, 1.1, 1.1, 1.1, 1.0, 1.0, 0.9, 0.9, 1.2, 1.2, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9, 0.7, 1.2, 1.2, 1.0, 0.9, 0.8, 0.7]  ↓ DÜŞÜŞ  ⚠️ ANORMAL
  Hemoglobin     : [16.2, 29.1, 31.9, 7.4, 7.3, 15.7, 16.1, 29.5, 33.4, 15.6, 29.2, 32.8, 29.5, 33.4, 15.7, 16.6, 29.8, 33.7, 32.8, 29.0, 15.5, 7.2, 16.3, 32.7, 29.9, 7.1, 33.1, 29.7, 15.3, 7.0, 32.8, 15.8, 28.5, 16.0, 29.7, 33.2, 7.7, 28.0, 15.4, 31.2, 7.9, 8.3, 31.5, 27.8, 14.6, 32.5, 15.0, 28.7, 7.6, 15.1, 27.1, 31.3, 7.3, 27.5, 15.0, 31.5, 14.7, 27.8, 31.8, 6.9, 15.7, 26.9, 32.8, 15.4, 27.0, 32.4, 7.7, 7.0, 14.8, 32.7, 27.8, 6.5, 14.2, 27.3, 32.6, 6.3, 27.9, 14.6, 32.8, 6.8, 14.9, 28.1, 33

---
## 💾 BÖLÜM 7: Bağlantıyı Kapat

In [20]:
con.close()
print(f'✅ DuckDB bağlantısı kapatıldı.')
print(f'   Veritabanı dosyası: {DB_PATH}')
db_size = os.path.getsize(DB_PATH) / (1024**3)
print(f'   DB boyutu: {db_size:.2f} GB')
print()
print('Sonraki adım: 03_sentinel_engine.ipynb')
print('  → Risk skoru motoru')
print('  → Kontrendikasyon kural tablosu')
print('  → NLP Agent entegrasyonu')

✅ DuckDB bağlantısı kapatıldı.
   Veritabanı dosyası: C:\Users\Mutlu\Desktop\ACUHIT\sentinel.db
   DB boyutu: 8.68 GB

Sonraki adım: 03_sentinel_engine.ipynb
  → Risk skoru motoru
  → Kontrendikasyon kural tablosu
  → NLP Agent entegrasyonu
